# Medical School Wellbeing — Scraping Pipeline

Each step saves an intermediate CSV so you can restart from any point.

1. Load university list
2. Discover subpages for each university
3. Scrape text from all subpages
4. Clean text (remove stopwords + punctuation)
5. Summary stats

# 0. Install all required packages

In [ ]:
import subprocess, sys

def install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

install("requests")
install("bs4")
install("pandas")
install("lxml")

# 1. Load university list

In [2]:
import os
import pandas as pd
from src.pipeline import (
    load_university_list,
    collect_all_subpages,
    scrape_all_pages,
    clean_all_texts,
)
from src.config import DATA_DIR

# Change this path to use a different university list CSV
CSV_PATH = os.path.join(DATA_DIR, "med_list.csv")

# Load university list
universities = load_university_list(csv_path=CSV_PATH)
print(f"Loaded {len(universities)} universities")
universities.head()

Loaded 65 universities


,Medical_School,Website
0,Stanford Medicine,https://med.stanford.edu/md.html
1,NYU Grossman School of Medicine,https://med.nyu.edu
2,Harvard Medical School,https://meded.hms.harvard.edu/md-program-landing
3,UC Davis School of Medicine,https://health.ucdavis.edu/mdprogram/
4,Vanderbilt University School of Medicine,https://medschool.vanderbilt.edu/md-program/


# 2. Get all subpages for each university

In [ ]:
subpages = collect_all_subpages(universities)
subpages.to_csv(os.path.join(DATA_DIR, "subpages.csv"), index=False)
print(f"\nSaved {len(subpages)} subpage URLs to Data/subpages.csv")
subpages.head()

# 3. Scrap text

In [ ]:
# Step 2: Scrape text from all subpages
# subpages = pd.read_csv(os.path.join(DATA_DIR, "subpages.csv"))

scraped = scrape_all_pages(subpages)
scraped.to_csv(os.path.join(DATA_DIR, "scraped_texts.csv"), index=False)
print(f"\nSaved scraped texts to Data/scraped_texts.csv")

# 4. Clean text

In [ ]:
# Step 3: Clean text
# scraped = pd.read_csv(os.path.join(DATA_DIR, "scraped_texts.csv"))

cleaned = clean_all_texts(scraped)
cleaned.to_csv(os.path.join(DATA_DIR, "cleaned_output.csv"), index=False)
print(f"\nSaved cleaned output to Data/cleaned_output.csv")

# 5. Basic summary states

In [ ]:
print(f"Universities: {cleaned['Medical_School'].nunique()}")
print(f"Total subpages scraped: {len(cleaned)}")
print(f"Pages with content: {(cleaned['Word_Count_Clean'] > 0).sum()}")
print(f"Avg clean word count per page: {cleaned['Word_Count_Clean'].mean():.0f}")
print()

# Per-school summary
school_stats = cleaned.groupby("Medical_School").agg(
    Pages=("Subpage_URL", "count"),
    Total_Words=("Word_Count_Clean", "sum"),
    Avg_Words=("Word_Count_Clean", "mean"),
).sort_values("Total_Words", ascending=False)

school_stats.head(10)